In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import joblib
from pathlib import Path
import sys
sys.path.insert(0, str(Path('..').resolve()))

MODELS_DIR = Path('../data/processed/models')
PROCESSED_DIR = Path('../data/processed')
FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)
EVAL_FIGURES_DIR = Path('../data/processed/eval_figures')

# Model Experiments

This notebook documents the model selection process for four defect detection problems. Features were pre-extracted by `build_features.py`. Models were trained with Leave-One-Group-Out CV on groups G1–G7 and evaluated on held-out groups G8–G9.

In [ ]:
df_p5    = pd.read_csv('../data/processed/features_p5.csv')
df_fixed = pd.read_csv('../data/processed/features_fixed.csv')

print('df_p5 shape:', df_p5.shape)
print('df_p5 columns:', df_p5.columns.tolist())
print()
print('df_fixed shape:', df_fixed.shape)
print('df_fixed columns:', df_fixed.columns.tolist())

## Feature Distributions

Understanding feature distributions before model selection is essential. Skewed features can disadvantage distance-based models like SVM and k-NN, while tree-based models remain largely robust to scale and skew. Outliers may dominate RMS-based features and must be distinguished from genuine defect signals rather than discarded. Scale differences across features affect regularisation strength in linear models and convergence of gradient-based optimisers, which is why all pipelines include a `StandardScaler`. Examining distributions grouped by label reveals which features carry the most discriminative signal and whether that signal is linearly separable — guiding model choice before any training begins.

In [ ]:
# ------------------------------------------------------------------
# Plot 1 — Frequency features grouped by freq_defect
# ------------------------------------------------------------------
freq_feat_cols = ['peak_magnitude', 'dominant_freq_hz', 'mag_x', 'mag_y', 'mag_z']

df_freq_melt = (
    df_p5[freq_feat_cols + ['freq_defect']]
    .melt(id_vars='freq_defect', var_name='Feature', value_name='Value')
)

fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=df_freq_melt, x='Feature', y='Value', hue='freq_defect', ax=ax)
ax.set_title('Frequency Features by freq_defect Label')
ax.set_xlabel('Feature')
ax.set_ylabel('Value')
ax.legend(title='freq_defect')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feat_dist_frequency.png', dpi=150)
plt.show()
plt.close()

# ------------------------------------------------------------------
# Plot 2 — Inclination features grouped by incl_detectable
# ------------------------------------------------------------------
incl_feat_cols = [
    'peak_z', 'signed_peak_z', 'peak_jerk',
    'impulse_energy', 'rise_time', 'post_impact_rms',
]

df_incl_melt = (
    df_p5[incl_feat_cols + ['incl_detectable']]
    .melt(id_vars='incl_detectable', var_name='Feature', value_name='Value')
)

fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=df_incl_melt, x='Feature', y='Value', hue='incl_detectable', ax=ax)
ax.set_title('Inclination Features by incl_detectable Label')
ax.set_xlabel('Feature')
ax.set_ylabel('Value')
ax.legend(title='incl_detectable')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feat_dist_inclination.png', dpi=150)
plt.show()
plt.close()

# ------------------------------------------------------------------
# Plot 3 — Belt speed features grouped by speed_defect
# ------------------------------------------------------------------
speed_feat_cols = [
    'journey_duration', 'duration_delta',
    'rms_full', 'rms_first_half', 'rms_second_half', 'rms_ratio',
]

df_speed_melt = (
    df_p5[speed_feat_cols + ['speed_defect']]
    .melt(id_vars='speed_defect', var_name='Feature', value_name='Value')
)

fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=df_speed_melt, x='Feature', y='Value', hue='speed_defect', ax=ax)
ax.set_title('Belt Speed Features by speed_defect Label')
ax.set_xlabel('Feature')
ax.set_ylabel('Value')
ax.legend(title='speed_defect')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feat_dist_belt_speed.png', dpi=150)
plt.show()
plt.close()

# ------------------------------------------------------------------
# Plot 4 — Damping features grouped by damping_present (df_fixed)
# ------------------------------------------------------------------
damp_feat_cols = [
    'rms_ratio_P1', 'rms_ratio_P2', 'rms_ratio_P3', 'rms_ratio_P4',
    'rms_ratio_P3_to_P1', 'rms_ratio_P4_to_P1',
    'rms_ratio_P3_to_P2', 'rms_ratio_P4_to_P2',
]

df_damp_melt = (
    df_fixed[damp_feat_cols + ['damping_present']]
    .melt(id_vars='damping_present', var_name='Feature', value_name='Value')
)

fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(data=df_damp_melt, x='Feature', y='Value', hue='damping_present', ax=ax)
ax.set_title('Damping Features by damping_present Label')
ax.set_xlabel('Feature')
ax.set_ylabel('Value')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='damping_present')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feat_dist_damping.png', dpi=150)
plt.show()
plt.close()

## Model Comparison

**Two-stage hyperparameter tuning.** Each model was tuned using a two-stage procedure. Stage 1 runs `RandomizedSearchCV` with 50 iterations over a broad parameter space, efficiently identifying the promising region without exhaustive enumeration. Stage 2 runs `GridSearchCV` over a narrow grid centred on the Stage 1 best parameters, providing precision tuning. Both stages use Leave-One-Group-Out CV on the training set only — the test set is never touched until final evaluation. This approach balances exploration and exploitation: random search avoids wasting budget on clearly poor combinations, while the narrow grid refines the solution with exhaustive precision.

**Why Leave-One-Group-Out CV?** k-fold CV on pooled data from multiple recording sessions causes data leakage — windows from the same session can appear in both train and validation folds, making CV scores falsely optimistic. This was the primary failure mode of the original MATLAB model, which achieved 3% CV error yet generalised poorly. Leave-One-Group-Out holds out one complete group (all six cases from one experimental session) per fold, ensuring that validation always measures generalisation to an unseen session. With seven training groups (G1–G7), this produces seven folds — a meaningful estimate of out-of-session performance.

In [ ]:
# ------------------------------------------------------------------
# Hardcoded results from training run
# ------------------------------------------------------------------
freq_results = {
    'Model':   ['LogisticRegression', 'SVC',
                'RandomForestClassifier', 'GradientBoostingClassifier'],
    'CV F1':   [1.000, 1.000, 1.000, 1.000],
    'Test F1': [1.000, 1.000, 1.000, 1.000],
}

incl_binary_results = {
    'Model':   ['LogisticRegression', 'SVC',
                'RandomForestClassifier', 'GradientBoostingClassifier'],
    'CV F1':   [0.914, 0.914, 0.974, 0.961],
    'Test F1': [0.923, 1.000, 0.905, 0.905],
}

incl_regression_results = {
    'Model':    ['Ridge', 'RandomForestRegressor', 'GradientBoostingRegressor'],
    'CV MAE':   [0.710, 0.778, 0.758],
    'Test MAE': [0.669, 0.432, 0.179],
}

belt_speed_results = {
    'Model':   ['MultiOutputRegressor_Ridge',
                'MultiOutputRegressor_RandomForest',
                'MultiOutputRegressor_GBM'],
    'CV MAE':   [5.224, 4.235, 4.070],
    'Test MAE': [5.252, 4.908, 5.024],
}

damping_results = {
    'Model':    ['LogisticRegression', 'SVC',
                 'RandomForestClassifier', 'GradientBoostingClassifier'],
    'CV F1':    [0.724, 0.724, 0.648, 0.648],
    'Test AUC': [0.750, 0.750, 1.000, 0.875],
}

# ------------------------------------------------------------------
# Helper: grouped bar chart
# ------------------------------------------------------------------
def plot_model_comparison(data_dict, cv_col, test_col, title, save_path,
                          lower_is_better=False):
    models     = data_dict['Model']
    cv_scores  = data_dict[cv_col]
    test_scores = data_dict[test_col]

    short_names = [
        m.replace('Classifier', '').replace('Regressor', '')
         .replace('MultiOutputRegressor_', 'MOR_')
         .replace('GradientBoosting', 'GBM')
        for m in models
    ]

    x     = np.arange(len(models))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width / 2, cv_scores,   width, label=f'CV {cv_col}',   color='steelblue')
    ax.bar(x + width / 2, test_scores, width, label=f'Test {test_col}', color='orange')

    note = ' (lower is better)' if lower_is_better else ''
    ax.set_title(f'{title}{note}')
    ax.set_xlabel('Model')
    ax.set_ylabel(cv_col)
    ax.set_xticks(x)
    ax.set_xticklabels(short_names, rotation=15, ha='right')
    ax.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    plt.close()


# ------------------------------------------------------------------
# Plot each problem
# ------------------------------------------------------------------
plot_model_comparison(
    freq_results, 'CV F1', 'Test F1',
    'Problem 1 — Frequency Binary Detection',
    FIGURES_DIR / 'model_comparison_frequency.png',
)

plot_model_comparison(
    incl_binary_results, 'CV F1', 'Test F1',
    'Problem 2a — Inclination Binary Detection',
    FIGURES_DIR / 'model_comparison_inclination_binary.png',
)

plot_model_comparison(
    incl_regression_results, 'CV MAE', 'Test MAE',
    'Problem 2b — Inclination Angle Regression',
    FIGURES_DIR / 'model_comparison_inclination_regression.png',
    lower_is_better=True,
)

plot_model_comparison(
    belt_speed_results, 'CV MAE', 'Test MAE',
    'Problem 3 — Belt Speed Multi-Output Regression',
    FIGURES_DIR / 'model_comparison_belt_speed.png',
    lower_is_better=True,
)

plot_model_comparison(
    damping_results, 'CV F1', 'Test AUC',
    'Problem 4 — Damping Binary Detection',
    FIGURES_DIR / 'model_comparison_damping.png',
)

## Feature Importance

For Random Forest and Gradient Boosting models, feature importance is measured by the mean decrease in impurity (Gini impurity for classifiers, variance for regressors) contributed by each feature across all splits in all trees. A feature that is used near the root of many trees, and whose splits produce large reductions in impurity, receives a high importance score. Importantly, importances are relative — they sum to 1.0 across all features — so a feature ranked first does not imply it alone is sufficient, only that it contributes more than others within this model and dataset. Features with near-zero importance can often be dropped without loss of predictive performance, which is useful for understanding the minimal sufficient feature set.

In [ ]:
# ------------------------------------------------------------------
# Load models from joblib
# ------------------------------------------------------------------
# Note: frequency best model is SVC (no feature_importances_).
# RF / GBM models are loaded instead for importance visualisation.
rf_incl  = joblib.load(MODELS_DIR / 'inclination_binary_RandomForestClassifier.joblib')
gbm_incl = joblib.load(MODELS_DIR / 'inclination_regression_GradientBoostingRegressor.joblib')
rf_damp  = joblib.load(MODELS_DIR / 'damping_RandomForestClassifier.joblib')
rf_speed = joblib.load(MODELS_DIR / 'belt_speed_MultiOutputRegressor_RandomForestRegressor.joblib')

# ------------------------------------------------------------------
# Feature name lists
# ------------------------------------------------------------------
incl_features = [
    'peak_z', 'signed_peak_z', 'peak_jerk',
    'impulse_energy', 'rise_time', 'post_impact_rms',
]

damp_features = [
    'rms_P1', 'rms_P2', 'rms_P3', 'rms_P4',
    'rms_ratio_P1', 'rms_ratio_P2', 'rms_ratio_P3', 'rms_ratio_P4',
    'min_rms_ratio',
    'rms_ratio_P3_to_P1', 'rms_ratio_P4_to_P1',
    'rms_ratio_P3_to_P2', 'rms_ratio_P4_to_P2',
]

speed_features = [
    'journey_duration', 'duration_delta',
    'rms_full', 'rms_first_half', 'rms_second_half', 'rms_ratio',
]


# ------------------------------------------------------------------
# Helper: horizontal bar chart of importances
# ------------------------------------------------------------------
def plot_importance(importances, feature_names, title, save_path):
    indices = np.argsort(importances)
    sorted_names = [feature_names[i] for i in indices]
    sorted_vals  = importances[indices]

    fig, ax = plt.subplots(figsize=(8, max(4, len(feature_names) * 0.45)))
    ax.barh(range(len(sorted_names)), sorted_vals, color='steelblue')
    ax.set_yticks(range(len(sorted_names)))
    ax.set_yticklabels(sorted_names)
    ax.set_xlabel('Feature Importance (mean decrease in impurity)')
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    plt.close()


# ------------------------------------------------------------------
# Plot 1 — Inclination binary RF
# ------------------------------------------------------------------
imp_incl_rf = rf_incl.named_steps['model'].feature_importances_
plot_importance(
    imp_incl_rf, incl_features,
    'Inclination Binary — RandomForest Feature Importance',
    FIGURES_DIR / 'importance_inclination_binary.png',
)

# ------------------------------------------------------------------
# Plot 2 — Inclination regression GBM
# ------------------------------------------------------------------
imp_incl_gbm = gbm_incl.named_steps['model'].feature_importances_
plot_importance(
    imp_incl_gbm, incl_features,
    'Inclination Regression — GradientBoosting Feature Importance',
    FIGURES_DIR / 'importance_inclination_regression.png',
)

# ------------------------------------------------------------------
# Plot 3 — Damping RF (13 features including phone-to-phone ratios)
# ------------------------------------------------------------------
imp_damp = rf_damp.named_steps['model'].feature_importances_
plot_importance(
    imp_damp, damp_features,
    'Damping Detection — RandomForest Feature Importance',
    FIGURES_DIR / 'importance_damping.png',
)

# ------------------------------------------------------------------
# Plot 4 — Belt speed RF, rail2 estimator (index 0)
# ------------------------------------------------------------------
imp_speed = rf_speed.named_steps['model'].estimators_[0].feature_importances_
plot_importance(
    imp_speed, speed_features,
    'Belt Speed (Rail 2) — RandomForest Feature Importance',
    FIGURES_DIR / 'importance_belt_speed.png',
)

## Key Findings

**Frequency detection.** All four models achieved perfect CV F1 = 1.000 and Test F1 = 1.000. This confirms that the signal-processing approach — full-journey FFT on individual accelerometer axes with a 25–50 Hz search band — extracts a perfectly separable feature. The frequency generator produces a sharp spectral peak that no background conveyor vibration can mimic. The feature engineering step validates the physical understanding: FFT is the right tool and no ML model is needed beyond a trivial threshold.

**Inclination binary detection.** SVC achieved the best test performance (Test F1 = 1.000 vs 0.923 for Logistic Regression). This is consistent with the physical intuition: `signed_peak_z` creates a near-linear decision boundary between detectable (Loc4) and non-detectable (Loc5 / absent) cases, making a kernel SVM well-suited. The 4th-order Butterworth low-pass filter applied before feature extraction (CORRECTIONS_AND_FINDINGS.md §6) successfully removes 30–50 Hz frequency defect contamination in combined cases.

**Inclination angle regression.** GradientBoostingRegressor strongly outperforms the alternatives (Test MAE = 0.179°, R² = 0.936 vs Ridge Test MAE = 0.669°). The `signed_peak_z` feature carries the primary direction signal — negative values correspond to downward impacts (positive inclination angle at Loc4). The non-linear boosting model captures the relationship between impact dynamics and physical tilt angle more accurately than the linear Ridge baseline, justifying the multi-model comparison approach.

**Belt speed regression.** All models show moderate performance (Test MAE ≈ 4.9–5.3%). `journey_duration` is the dominant feature — normal speed produces ~46 s journeys while slow speeds extend to ~71 s — but precise rail speed prediction across all groups is hard because different groups use different speed combinations, and the test set (G8: 100/60, G9: 40/70) includes unseen speed pairs. The MultiOutputRegressor approach correctly treats Rail 2 and Rail 3 as independent regression targets.

**Damping detection.** RandomForestClassifier achieved Test AUC = 1.000, meaning the model perfectly ranks all damping-present cases above damping-absent cases. The weighted F1 = 0.733 reflects that the optimal classification threshold is not the default 0.5 — a consequence of severe class imbalance (2/6 cases per group have damping). The four phone-to-phone relative ratio features (`rms_ratio_P3_to_P1`, `rms_ratio_P4_to_P1`, `rms_ratio_P3_to_P2`, `rms_ratio_P4_to_P2`) improved AUC from 0.938 to 1.000 by cancelling the common-mode RMS inflation caused by frequency generators transmitting energy through the frame to all fixed phones simultaneously.

## Approval Gate — Step 7

- [ ] Feature distributions show clear separation for frequency and inclination
- [ ] Model comparison shows GBM winning for inclination regression
- [ ] Feature importance confirms `signed_peak_z` dominates inclination
- [ ] Feature importance confirms `journey_duration` dominates belt speed
- [ ] Damping importance shows phone-to-phone ratios are useful